In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # if notebook is inside a subfolder

import pandas as pd
from src.pipeline import pipeline    
from sklearn.model_selection import train_test_split

import joblib

In [2]:
import pandas as pd 

#import datasets
df = pd.read_csv("../datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")

#divide data train and test
X = df.drop('Churn', axis=1)
y = df['Churn']
y=y.map({'No':0,'Yes':1})
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=5295,
    stratify=y
)


In [3]:
pipeline.fit(X_train,y_train)

,steps,"[('feature_engineering', ...), ('preprocessing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('power', ...), ('scale', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a s

In [4]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(pipeline, "../models/churn_pipeline.pkl")

['../models/churn_pipeline.pkl']

In [8]:
import joblib
from pathlib import Path

project_root = Path().resolve().parent
model_path = project_root / "models"

model_path.mkdir(exist_ok=True)

joblib.dump(pipeline, model_path / "churn_pipeline.pkl")

['C:\\Users\\Cheta\\Desktop\\project-2-Telcom_Customer_Churn_Analysis\\models\\churn_pipeline.pkl']

In [5]:
import joblib

pipeline = joblib.load("../models/churn_pipeline.pkl")  #to load model

In [11]:
pipeline.predict(X_test)

array([1, 0, 0, ..., 0, 1, 0], shape=(1409,))

In [10]:
int(pipeline.predict(X_test.iloc[[0]])[0])

1

In [23]:
round(max(pipeline.predict_proba(X_test.iloc[[0]])[0])*100,2)

np.float64(88.37)

In [35]:
import shap

X_fe = pipeline.named_steps["feature_engineering"].transform(X_test.iloc[[0]])
X_pre = pipeline.named_steps["preprocessing"].transform(X_fe)

model = pipeline.named_steps["model"]

import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_pre)
shap_values
#to get feature name

feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()
feature_names

array(['power__TotalCharges', 'scale__MonthlyCharges', 'scale__tenure',
       'binary__PhoneService', 'binary__PaperlessBilling',
       'onehot__InternetService_DSL',
       'onehot__InternetService_Fiber optic',
       'onehot__InternetService_No', 'onehot__Contract_Month-to-month',
       'onehot__Contract_One year', 'onehot__Contract_Two year',
       'onehot__PaymentMethod_Bank transfer (automatic)',
       'onehot__PaymentMethod_Credit card (automatic)',
       'onehot__PaymentMethod_Electronic check',
       'onehot__PaymentMethod_Mailed check', 'remainder__gender',
       'remainder__SeniorCitizen', 'remainder__Partner',
       'remainder__Dependents', 'remainder__MultipleLines',
       'remainder__OnlineSecurity', 'remainder__OnlineBackup',
       'remainder__DeviceProtection', 'remainder__TechSupport',
       'remainder__StreamingTV', 'remainder__StreamingMovies',
       'remainder__MonthlyCharges_category'], dtype=object)

In [36]:
shap_values

array([[[-0.03066663,  0.03066663],
        [-0.00145645,  0.00145645],
        [-0.09321564,  0.09321564],
        [ 0.00016351, -0.00016351],
        [-0.0078554 ,  0.0078554 ],
        [-0.01406519,  0.01406519],
        [-0.03879441,  0.03879441],
        [-0.00522666,  0.00522666],
        [-0.0647788 ,  0.0647788 ],
        [-0.0066984 ,  0.0066984 ],
        [-0.01982314,  0.01982314],
        [-0.00045639,  0.00045639],
        [-0.00131167,  0.00131167],
        [-0.02573814,  0.02573814],
        [-0.00019528,  0.00019528],
        [-0.00177069,  0.00177069],
        [ 0.00133888, -0.00133888],
        [-0.00272576,  0.00272576],
        [-0.00215533,  0.00215533],
        [ 0.00521124, -0.00521124],
        [-0.04684922,  0.04684922],
        [-0.02177025,  0.02177025],
        [-0.00728247,  0.00728247],
        [-0.04564494,  0.04564494],
        [-0.00159091,  0.00159091],
        [-0.00692403,  0.00692403],
        [ 0.00083016, -0.00083016]]])

In [42]:
shap_values[0]

array([[-0.03066663,  0.03066663],
       [-0.00145645,  0.00145645],
       [-0.09321564,  0.09321564],
       [ 0.00016351, -0.00016351],
       [-0.0078554 ,  0.0078554 ],
       [-0.01406519,  0.01406519],
       [-0.03879441,  0.03879441],
       [-0.00522666,  0.00522666],
       [-0.0647788 ,  0.0647788 ],
       [-0.0066984 ,  0.0066984 ],
       [-0.01982314,  0.01982314],
       [-0.00045639,  0.00045639],
       [-0.00131167,  0.00131167],
       [-0.02573814,  0.02573814],
       [-0.00019528,  0.00019528],
       [-0.00177069,  0.00177069],
       [ 0.00133888, -0.00133888],
       [-0.00272576,  0.00272576],
       [-0.00215533,  0.00215533],
       [ 0.00521124, -0.00521124],
       [-0.04684922,  0.04684922],
       [-0.02177025,  0.02177025],
       [-0.00728247,  0.00728247],
       [-0.04564494,  0.04564494],
       [-0.00159091,  0.00159091],
       [-0.00692403,  0.00692403],
       [ 0.00083016, -0.00083016]])

In [44]:
shap_values_class1 = shap_values[0][:,1]
shap_values_class0 = shap_values[0][:,0]

In [ ]:
pd